# TP2 – Exercice 1 : Reconnaissance de chiffres manuscrits (MNIST)
**Module :** Technologies IA – ENSA Fès – Prof. Youness IDRISSI KHAMLICHI

Objectif : construire un réseau de neurones entièrement connecté (Dense) avec TensorFlow/Keras pour reconnaître les chiffres 0–9.

## 1. Import des bibliothèques

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Flatten
from tensorflow.keras.datasets import mnist
import matplotlib.pyplot as plt
import numpy as np

print("TensorFlow version:", tf.__version__)

### Réponses
1. **TensorFlow vs Keras** : TensorFlow est le moteur de calcul (tenseurs, CPU/GPU, différentiation automatique). Keras est l'API haut niveau (`tf.keras`) qui simplifie la construction et l'entraînement des réseaux.
2. **Flatten** : transforme chaque image 2D (28×28) en un vecteur 1D de 784 valeurs, nécessaire avant les couches Dense.
3. **Version de TensorFlow** : affichée par la cellule ci-dessus (`tf.__version__`).
4. **Sequential / Dense** : `Sequential` empile les couches linéairement. `Dense` est une couche entièrement connectée (chaque neurone reçoit toutes les sorties de la couche précédente).

## 2. Chargement et préparation du dataset

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("Type des données :", x_train.dtype)
x_train = x_train / 255.0
x_test = x_test / 255.0

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape:", x_test.shape)

### Réponses
1. **Normalisation** : ramener les pixels de [0, 255] à [0, 1] stabilise les gradients et accélère la convergence.
2. **Taille d'une image** : 28×28 pixels, niveaux de gris. Type d'origine : `uint8` (0–255), devenu `float64` après la division.
3. **Exemples d'entraînement** : 60 000 (et 10 000 pour le test).

## 3. Visualisation d'échantillons

In [ ]:
plt.figure(figsize=(8, 3))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap='gray')
    plt.title(f"Label: {y_train[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

### Réponses
1. Pour changer le nombre d'images : modifier `range(10)` et la grille `plt.subplot(lignes, colonnes, i+1)`.
2. `cmap='gray'` : affiche l'image en niveaux de gris (sinon palette de couleurs par défaut).
3. Cette étape permet de vérifier que les données et les labels sont corrects avant d'entraîner (détecter erreurs de chargement, d'étiquetage, de format).

## 4. Construction du modèle

In [ ]:
model = Sequential([
    Input(shape=(28, 28)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

model.summary()

### Réponses
1. **Paramètres** : 784×128 + 128 = 100 480 (couche cachée) et 128×10 + 10 = 1 290 (sortie), soit **101 770 paramètres**.
2. **Activation de sortie** : `softmax`, car elle produit une distribution de probabilités sur les 10 classes (somme = 1).
3. **ReLU vs Softmax** : ReLU = max(0, x), non linéarité utilisée dans les couches cachées. Softmax normalise un vecteur en probabilités, utilisée en sortie pour la classification multiclasse.

## 5. Compilation

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

### Réponses
1. **Optimiseur** : algorithme (Adam, SGD, RMSprop…) qui met à jour les poids à partir des gradients pour réduire la perte.
2. **Fonction de perte** : mesure l'écart entre prédiction et vérité ; c'est la quantité que l'on minimise.
3. **Changer la perte** : change l'objectif d'optimisation ; une perte inadaptée (ex. MSE pour de la classification, ou `categorical_crossentropy` avec des labels entiers) donne de mauvais résultats ou une erreur.
4. **Mauvais optimiseur** : convergence lente, instable (divergence) ou blocage dans un minimum local / plateau.
5. **loss vs metrics** : la loss est optimisée par l'entraînement (différentiable) ; les metrics (accuracy) servent uniquement à évaluer et à interpréter.

## 6. Entraînement

In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(x_test, y_test)
)

### Réponses
1. **Époque** : un passage complet sur tout le jeu d'entraînement.
2. **Jeu de validation** : mesurer la généralisation pendant l'entraînement et détecter le surapprentissage.
3. **50 époques** : la précision d'entraînement continue de monter, mais la validation stagne ou baisse (overfitting) ; il faut utiliser `EarlyStopping` ou de la régularisation.

## 7. Évaluation

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

### Réponses
1. **Précision finale** : voir la valeur affichée (généralement ≈ 97–98 %).
2. **Test < entraînement** : le modèle a optimisé ses poids sur les données d'entraînement ; les données de test sont nouvelles.
3. **Overfitting** : Dropout, régularisation L2, early stopping, plus de données / data augmentation, modèle plus simple.

## 8. Courbes de performance

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'], label='Précision entraînement')
plt.plot(history.history['val_accuracy'], label='Précision validation')
plt.title('Courbe de précision')
plt.xlabel('Époques')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Perte entraînement')
plt.plot(history.history['val_loss'], label='Perte validation')
plt.title('Courbe de perte')
plt.xlabel('Époques')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

### Réponses
1. Les deux courbes de précision montent et restent proches : le modèle apprend bien.
2. **Détection visuelle de l'overfitting** : la précision d'entraînement continue d'augmenter alors que la validation stagne ou baisse (et la perte de validation remonte).
3. **Remèdes** : Dropout, early stopping, régularisation, data augmentation.
4. **Courbes divergentes** : arrêter plus tôt, simplifier le modèle, ajouter de la régularisation, obtenir plus de données.

## 9. Prédictions

In [ ]:
predictions = model.predict(x_test)

plt.figure(figsize=(10, 4))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(x_test[i], cmap='gray')
    plt.title(f"Vrai: {y_test[i]}\nPrédit: {np.argmax(predictions[i])}")
    plt.axis('off')
plt.tight_layout()
plt.show()

# Images mal classées
errors = np.where(np.argmax(predictions, axis=1) != y_test)[0]
print("Nombre d'erreurs :", len(errors))
plt.figure(figsize=(10, 4))
for k, i in enumerate(errors[:5]):
    plt.subplot(1, 5, k + 1)
    plt.imshow(x_test[i], cmap='gray')
    plt.title(f"Vrai: {y_test[i]}\nPrédit: {np.argmax(predictions[i])}")
    plt.axis('off')
plt.tight_layout()
plt.show()

### Réponses
1. Oui, quelques erreurs, surtout sur des chiffres mal écrits ou ambigus (4/9, 3/5, 7/1).
2. Écriture atypique, bruit, formes proches ; un réseau Dense ignore la structure spatiale 2D.
3. Un **CNN** exploite la structure spatiale et améliore la précision (≈ 99 %).

## Analyse personnelle (à compléter après exécution)
- **Interprétation des résultats** : …
- **Difficultés rencontrées** : …
- **Propositions d'amélioration** : plus de couches, Dropout, autres activations, CNN, réglage des hyperparamètres (batch size, learning rate, nombre d'époques).